# 랭스미스 LangSmith

https://smith.langchain.com/

LangSmith는 OpenAI의 언어 모델(ChatGPT 등)을 활용하여 맞춤형 언어 애플리케이션을 개발하고 디버깅하기 위한 도구 및 플랫폼이다.

.
개발자가 자연어 처리 기반의 애플리케이션을 효과적으로 설계하고 관리할 수 있도록 다양한 기능을 제공한다.

1. **트레이싱 및 디버깅(Tracing & Debugging)**  
   - 생성된 응답을 추적하고, 언어 모델의 동작을 분석하여 결과를 최적화할 수 있다.
   - 디버깅 도구를 통해 애플리케이션의 흐름에서 문제를 쉽게 식별하고 수정 가능.

2. **모델 조정(Customization)**  
   - 기본 모델 외에도 프롬프트 설계(prompt engineering)와 파인튜닝(fine-tuning)을 활용하여 애플리케이션에 적합한 응답을 생성할 수 있다.

3. **평가 및 테스트(Evaluation & Testing)**  
   - 다양한 입력과 환경에서 모델의 성능을 테스트하고, 모델 결과를 체계적으로 비교 가능.
   - 지속적으로 성능을 측정해 품질을 유지.

4. **워크플로우 통합(Workflow Integration)**  
   - API 및 SDK를 제공하여 LangChain과 같은 프레임워크와 손쉽게 통합 가능.
   - 빠르고 직관적인 프로토타입 제작 지원.

5. **버전 관리(Versioning)**  
   - 모델 및 프롬프트 변경 사항을 체계적으로 기록하여 실험의 재현성과 관리 용이.


> LangChain으로 빌드하는 많은 애플리케이션은 LLM 호출을 여러 번 호출하는 아키텍처를 가질 수 있다.
>
> 이러한 애플리케이션이 점점 더 복잡해짐에 따라 체인이나 에이전트 내부에서 정확히 무슨 일이 일어나고 있는지를 검사하는 것이 중요해진다.
>
> 랭스미스(LangSmith)는 이러한 단계를 모니터링 할 수 있게 해주기 때문에 LLM 애플리케이션을 개발 할 때 랭스미스를 사용하는 것을 권장한다.

In [1]:
# install -U : 업그레이드 (설치돼있으면 최신버전으로, 없으면 최신버전 설치)
%pip install -U langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [12]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

def get_weather(city: str) -> str:
    """날씨 정보를 반환하는 함수"""
    return f"It's always sunny in {city}"

model = ChatOpenAI(
    MODEL = "gpt-5.6-luna",
    # none -> 추론 최소화, 빠른 응답 / low -> 가벼운 추론 / medium -> 중간 추론 / high -> 깊은 추론
    reasoning_effort = 'none' # 별도의 추론을 최소화해서 빠르게 응답
)

agent = create_agent(
    model = 'openai:gpt-5-mini',
    tools = [get_weather],
    system_prompt = 'Your are helpful assistant'
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "What is the weather in San Francisco?"}
    ]}
)

response   

c:\Users\SJ\OneDrive\Desktop\study\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3715: UserWarning: WARNING! MODEL is not default parameter.
                MODEL was transferred to model_kwargs.
                Please confirm that MODEL is what you intended.
  if await self.run_code(code, result, async_=asy):


{'messages': [HumanMessage(content='What is the weather in San Francisco?', additional_kwargs={}, response_metadata={}, id='c4e34f02-94ed-4a0c-990f-e50fc1cf0faa'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 141, 'total_tokens': 229, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EGFXzT4ptC8ixzebh1dWyjPcxRSCY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a031ca-07a2-7340-80a0-50054738ab8e-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_1sPWRMVffeO8TVpf0

In [13]:
# 사용자 질문 원본 : 'What is the weather in San Francisco'
response['messages'][0].content

'What is the weather in San Francisco?'

In [14]:
# LLM 답변 : 'The source I checked says: "It\'s always sunny in San Francisco.""\n\nWould you like a"
response['messages'][-1].content

'According to the weather service I queried: "It\'s always sunny in San Francisco."\n\nWould you like more details (current temperature, wind, hourly forecast, or a multi-day forecast)?'